# ITCHI v0.2 frente a v0.1: Cristóbal 2020 y Odile 2014

Este notebook muestra qué cambia ITCHI v0.2 y si el cambio mejora la correspondencia con las declaratorias.

| Experimento | Peligro por lluvia $H_P$ | Peligro por viento $H_W$ |
|---|---|---|
| `v0.1` | Rampa R3: Q90 → 0, Q95 → 0.5, Q99 → 1 | Rankine modificado, solo dentro de R34 |
| `R4` | Rampa R4: Q75 → 0, Q90 → 0.25, Q95 → 0.5, Q99 → 1 | Igual que v0.1 |
| `v0.2` | R4 escalada por persistencia (R24) y antecedente (A72) | W06 calibrado a R34, con cola hasta el radio de atribución |

En los tres experimentos el radio de atribución es el mismo, W06 con metodología PA21, R(V = 2 m/s), para aislar el efecto de lluvia y viento.

**Qué lluvia se usa.** En cada snapshot $T$ se usa la lluvia observada de MSWEP V3.16 de esos días: $P_6$ en $(T-6\,\mathrm{h}, T]$, $R_{24}$ en $(T-24\,\mathrm{h}, T]$ y $A_{72}$ en $(T-78\,\mathrm{h}, T-6\,\mathrm{h}]$. Cada una se compara con los percentiles climatológicos de su duración (2000–2025, mayo–noviembre), que son los umbrales.

**Verificación municipal.** Para cada municipio dentro de la zona de influencia se toma el máximo de `ITCHI_max` sobre sus píxeles, su clase SIAT, y se compara con la existencia de declaratoria asociada al ciclón (POD, POFD, PSS por umbral de clase). Son solo dos ciclones: sirve para ilustrar, no para calibrar.

## 0. Configuración

In [ ]:
import os
import re
import warnings
from pathlib import Path

import cartopy.crs as ccrs
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

import itchi.itchi_size_helpers as H
from itchi.aggregation import compute_event_max
from itchi.config import get_project_root, load_default_config
from itchi.constants import ITCHI_VERSION, QUADRANTS
from itchi.declaratorias_6h import municipio_pixels
from itchi.mswep import subset_mswep_domain
from itchi.pipeline import compute_itchi_snapshot_from_grid
from itchi.plotting import SIAT_DANGER_LABELS, classify_itchi_to_siat_levels, get_siat_hazard_cmap_norm
from itchi.precipitation import (
    RAMP_R4_NODES,
    compute_antecedent_precipitation,
    compute_precipitation_hazard_ramp,
    compute_precipitation_matrix_hazard,
)
from itchi.tracks import DEFAULT_R34_COLUMN_MAP

ROOT = get_project_root()
CFG = load_default_config()
print("itchi", ITCHI_VERSION, "|", ROOT)

In [ ]:
CASES = {
    "CRISTOBAL2020": dict(name="CRISTOBAL", season=2020, basin="NA", atcf="AL032020", t_map="2020-06-04 00:00"),
    "ODILE2014": dict(name="ODILE", season=2014, basin="EP", atcf="EP152014", t_map="2014-09-15 00:00"),
}


def local_path(path) -> Path:
    # "D:/..." -> "/mnt/d/..." en WSL/Linux; sin cambios en Windows
    s = str(path).replace("\\", "/")
    if os.name != "nt" and re.match(r"^[A-Za-z]:/", s):
        return Path(f"/mnt/{s[0].lower()}") / s[3:]
    return Path(s)


MSWEP_6H_DIR = local_path("D:/MSWEP_V316_6H")
CLIM = {d: ROOT / "data" / "processed" / f"mswep_v3_{d}h_percentiles_may_nov.nc" for d in (6, 24, 72)}
CLIM_VAR = "p_wet_raw"          # la misma variable con la que se calibró la escalada
CLIM_DOMAIN = dict(lon_min=-140.0, lon_max=-60.0, lat_min=5.0, lat_max=35.0)
PAD_DEG = 12.0

RAMPS = {"R3": {"Q90": 0.0, "Q95": 0.5, "Q99": 1.0}, "R4": RAMP_R4_NODES}
EXPERIMENTS = {
    "v0.1": dict(precip="R3", wind="v01"),
    "R4": dict(precip="R4", wind="v01"),
    "v0.2": dict(precip="matriz", wind="w06_tail"),
}
EXP_COLORS = {"v0.1": "#73726c", "R4": "#eb6834", "v0.2": "#2a78d6"}
FIELDS = ("ITCHI", "H_P", "H_W", "H_dir", "H_ind", "M_direct", "M_indirect")

DECL_FILE = ROOT / "outputs" / "diagnostics" / "declaratorias_6h" / "declaratorias_ct_eventos.csv"
MUN_SHP = ROOT / "data" / "external" / "mun23gw" / "mun23gw.shp"
FIG_DIR = ROOT / "outputs" / "figures" / "itchi_v02_mejoras"
FIG_DIR.mkdir(parents=True, exist_ok=True)

assert MSWEP_6H_DIR.is_dir(), f"no existe {MSWEP_6H_DIR}"
for d, f in CLIM.items():
    assert f.exists(), f"falta {f.name}: corre scripts/percentiles_6h.py --durations {d}"
SIAT_CMAP, SIAT_NORM = get_siat_hazard_cmap_norm()
PC = ccrs.PlateCarree()

## 1. Trayectoria, lluvia observada y climatología

`read_precip` lee la temporada del año desde 84 h antes del primer snapshot. Así $R_{24}$ y $A_{72}$ tienen sus ventanas completas desde el primer paso; ventanas incompletas quedan NaN y no escalan.

In [ ]:
def load_track(case: str) -> pd.DataFrame:
    c = CASES[case]
    ibtracs = ROOT / CFG["paths"]["tracks"] / f"IBTrACS.{c['basin']}.v04r01.nc"
    rocloud = ROOT / "data" / "external" / "Rocloud" / f"ROCLOUD_{c['basin']}.csv"
    track = H.load_case_track(ibtracs, rocloud, c["name"], c["season"], c["atcf"])
    ok = np.array([H.attribution_radii_km(row, "pa21")[0] is not None for _, row in track.iterrows()])
    return track[ok].reset_index(drop=True)


def case_bbox(track: pd.DataFrame) -> dict:
    return dict(
        lon_min=max(float(track["lon"].min()) - PAD_DEG, CLIM_DOMAIN["lon_min"]),
        lon_max=min(float(track["lon"].max()) + PAD_DEG, CLIM_DOMAIN["lon_max"]),
        lat_min=max(float(track["lat"].min()) - PAD_DEG, CLIM_DOMAIN["lat_min"]),
        lat_max=min(float(track["lat"].max()) + PAD_DEG, CLIM_DOMAIN["lat_max"]),
    )


def read_precip(track: pd.DataFrame, bbox: dict):
    # P6, R24 y A72 observados en los tiempos de la trayectoria
    t = pd.DatetimeIndex(track["time"])
    path = MSWEP_6H_DIR / f"mswep_v3_6h_{t[0].year}.nc"
    with xr.open_dataset(path) as ds:
        p6 = subset_mswep_domain(ds["precipitation"], **bbox)
        p6 = p6.sel(time=slice(t.min() - pd.Timedelta(hours=84), t.max())).load()
    missing = t.difference(pd.DatetimeIndex(p6["time"].values))
    assert missing.empty, f"tiempos sin acumulado de 6 h: {list(missing)}"
    r24 = compute_antecedent_precipitation(p6, hours=24, exclude_current=False)
    a72 = compute_antecedent_precipitation(p6, hours=72, exclude_current=True)
    return tuple(da.sel(time=t.values) for da in (p6, r24, a72))


def read_clim(d: int, template: xr.DataArray) -> dict:
    # Percentiles locales de la duración d en la malla de la lluvia: {"Q75": ..., "Q90": ...}
    with xr.open_dataset(CLIM[d]) as ds:
        q = ds[CLIM_VAR].load()
    q = q.assign_coords(quantile=[f"Q{v * 100:g}" for v in q["quantile"].values])
    q = q.reindex_like(template, method="nearest", tolerance=1e-3)
    return {k: q.sel(quantile=k, drop=True) for k in q["quantile"].values}

## 2. Peligro por lluvia en los tres experimentos

`matriz` es la v0.2: $H_P = (1-f)\,H_{R4} + f\,H_{esc}$, con $f = f_{R24}\,f_{A72}$. La rampa escalada lleva Q95 a 0.8. $f_{R24}$ va de 0 en Q90₂₄ a 1 en Q95₂₄ y $f_{A72}$ de 0 en 0 mm a 1 en Q75₇₂. Los píxeles sin percentiles quedan con $H_P = 0$, como en el notebook 03.

In [ ]:
def precip_hazards(p6, r24, a72, q6, q24, q72) -> dict:
    hp = {
        "R3": compute_precipitation_hazard_ramp(p6, q6, RAMPS["R3"]),
        "R4": compute_precipitation_hazard_ramp(p6, q6, RAMPS["R4"]),
        "matriz": compute_precipitation_matrix_hazard(
            p6, q6, {"R24": r24, "A72": a72}, {"R24": q24, "A72": q72}
        ),
    }
    return {k: v.fillna(0.0).transpose("time", "lat", "lon") for k, v in hp.items()}

## 3. ITCHI por snapshot y por experimento

Todo pasa por `compute_itchi_snapshot_from_grid` con control de calidad. En `v0.2` se usan `precipitation_hazard` y `wind_model="w06_tail"`, y el viento entra también al componente indirecto: $H_{ind} = 1-(1-H_{W,ind})(1-H_{P,ind})$.

In [ ]:
def run_case(case: str) -> dict:
    track = load_track(case)
    bbox = case_bbox(track)
    p6, r24, a72 = read_precip(track, bbox)
    template = p6.isel(time=0, drop=True)
    q6, q24, q72 = (read_clim(d, template) for d in (6, 24, 72))
    hp = precip_hazards(p6, r24, a72, q6, q24, q72)
    lat, lon = template["lat"].values, template["lon"].values
    lon2d, lat2d = np.meshgrid(lon, lat)

    events = {}
    for exp, spec in EXPERIMENTS.items():
        fields = {k: [] for k in FIELDS}
        for i, row in track.iterrows():
            radii, _ = H.attribution_radii_km(row, "pa21")
            rmw = float(row["rmw_km"]) if np.isfinite(row["rmw_km"]) else float(row["pa21_rmax_km"])
            fallback = float(row["pa21_r34_km"]) if np.isfinite(row["pa21_r34_km"]) else None
            res = compute_itchi_snapshot_from_grid(
                p6.values[i], None, None, None,
                precipitation_hazard=hp[spec["precip"]].values[i],
                lon=lon2d, lat=lat2d, center_lon=float(row["lon"]), center_lat=float(row["lat"]),
                r34_by_quadrant={q: row[DEFAULT_R34_COLUMN_MAP[q]] for q in QUADRANTS}, r34_unit="nm",
                rocloud_by_quadrant=radii, rocloud_unit="km",
                vmax_kt=float(row["vmax_kt"]), rmw_km=rmw, fallback_direct_radius_km=fallback,
                wind_model=spec["wind"], run_quality_control=True,
            )
            for k in FIELDS:
                fields[k].append(np.asarray(res[k], dtype=np.float32))
        ds = xr.Dataset({k: (("time", "lat", "lon"), np.stack(v)) for k, v in fields.items()},
                        coords={"time": track["time"].values, "lat": lat, "lon": lon})
        ds["ITCHI_max"] = compute_event_max(ds["ITCHI"], dim="time")
        events[exp] = ds
    return dict(track=track, bbox=bbox, p6=p6, r24=r24, a72=a72, q6=q6, q24=q24, q72=q72,
                hp=hp, events=events, lon2d=lon2d, lat2d=lat2d)


OUT = {}
for case in CASES:
    OUT[case] = run_case(case)
    o = OUT[case]
    print(f"{case}: {len(o['track'])} snapshots | malla {o['lon2d'].shape} | "
          + " | ".join(f"{e} ITCHI_max medio {float(ds['ITCHI_max'].where(ds['ITCHI_max'] > 0).mean()):.3f}"
                       for e, ds in o["events"].items()))

## 4. Mapas del evento: clase SIAT de `ITCHI_max`

Las clases salen de `classify_itchi_to_siat_levels`, en tramos de 0.2. La v0.2 cambia el mapa por tres vías: R4 activa la clase baja desde P75, la escalada sube la lluvia intensa y persistente sobre suelo húmedo, y la cola de viento agrega peligro bajo entre R34 y el radio de atribución.

In [ ]:
def siat_levels(field) -> np.ndarray:
    lv = classify_itchi_to_siat_levels(np.asarray(field, dtype=float))
    return np.where(lv > 0, lv, np.nan)


def map_axes(o: dict, ncols: int, nrows: int = 1, size=(5.2, 4.2)):
    fig, axes = plt.subplots(nrows, ncols, figsize=(size[0] * ncols, size[1] * nrows),
                             subplot_kw={"projection": PC}, squeeze=False)
    b = o["bbox"]
    for ax in axes.ravel():
        ax.set_extent([b["lon_min"], b["lon_max"], b["lat_min"], b["lat_max"]], crs=PC)
        ax.coastlines(lw=0.6)
        ax.plot(o["track"]["lon"], o["track"]["lat"], "k.-", lw=0.8, ms=2.5, transform=PC)
    return fig, axes


for case, o in OUT.items():
    fig, axes = map_axes(o, len(EXPERIMENTS))
    for ax, (exp, ds) in zip(axes[0], o["events"].items()):
        m = ax.pcolormesh(o["lon2d"], o["lat2d"], siat_levels(ds["ITCHI_max"]), cmap=SIAT_CMAP,
                          norm=SIAT_NORM, transform=PC, shading="auto")
        ax.set_title(f"{exp}", fontsize=11)
    cb = fig.colorbar(m, ax=axes.ravel().tolist(), shrink=0.75, ticks=np.arange(6) + 0.5)
    cb.set_ticklabels(SIAT_DANGER_LABELS)
    fig.suptitle(f"{case}: clase SIAT de ITCHI_max", fontsize=12)
    fig.savefig(FIG_DIR / f"{case.lower()}_siat_3exp.png", dpi=200, bbox_inches="tight")

## 5. Dónde cambia el índice: diferencias de `ITCHI_max`

In [ ]:
for case, o in OUT.items():
    ev = o["events"]
    pairs = [("v0.2", "v0.1"), ("R4", "v0.1"), ("v0.2", "R4")]
    fig, axes = map_axes(o, len(pairs))
    for ax, (a, b) in zip(axes[0], pairs):
        d = (ev[a]["ITCHI_max"] - ev[b]["ITCHI_max"]).values
        m = ax.pcolormesh(o["lon2d"], o["lat2d"], np.where(np.abs(d) > 1e-6, d, np.nan), cmap="RdBu_r",
                          vmin=-0.6, vmax=0.6, transform=PC, shading="auto")
        ax.set_title(f"{a} − {b}", fontsize=11)
    fig.colorbar(m, ax=axes.ravel().tolist(), shrink=0.75, label="Δ ITCHI_max")
    fig.suptitle(f"{case}: rojo = v0.2 o R4 marcan más peligro", fontsize=12)
    fig.savefig(FIG_DIR / f"{case.lower()}_diferencias.png", dpi=200, bbox_inches="tight")

## 6. Un snapshot por dentro: lluvia, escalada y viento

En el tiempo `t_map` de cada caso:
- $P_6$;
- el peso de escalada visto como $H_P^{v0.2} - H_P^{R4}$, que solo es positivo donde la lluvia es intensa, persiste y cae sobre suelo húmedo;
- $H_W$ de la v0.1, que llega a cero en R34;
- $H_W$ de la v0.2, con la cola.

In [ ]:
for case, o in OUT.items():
    t = pd.Timestamp(CASES[case]["t_map"])
    i = int(np.where(pd.DatetimeIndex(o["track"]["time"]) == t)[0][0])
    panels = [
        ("P6 (mm en 6 h)", o["p6"].values[i], dict(cmap="Blues", vmin=0, vmax=80)),
        ("H_P v0.2 − H_P R4 (escalada)", (o["hp"]["matriz"] - o["hp"]["R4"]).values[i], dict(cmap="Purples", vmin=0, vmax=0.3)),
        ("H_W v0.1 (solo dentro de R34)", o["events"]["v0.1"]["H_W"].values[i], dict(cmap="Oranges", vmin=0, vmax=1)),
        ("H_W v0.2 (con cola hasta R_W06)", o["events"]["v0.2"]["H_W"].values[i], dict(cmap="Oranges", vmin=0, vmax=1)),
    ]
    fig, axes = map_axes(o, 2, 2)
    for ax, (title, field, kw) in zip(axes.ravel(), panels):
        m = ax.pcolormesh(o["lon2d"], o["lat2d"], np.where(field > 0, field, np.nan), transform=PC,
                          shading="auto", **kw)
        ax.plot(o["track"]["lon"][i], o["track"]["lat"][i], "k*", ms=10, transform=PC)
        fig.colorbar(m, ax=ax, shrink=0.8)
        ax.set_title(title, fontsize=10)
    fig.suptitle(f"{case}, {t:%Y-%m-%d %H} UTC", fontsize=12)
    fig.savefig(FIG_DIR / f"{case.lower()}_snapshot_componentes.png", dpi=200, bbox_inches="tight")

### Perfil radial del peligro por viento en ese snapshot

In [ ]:
def haversine_km(lat0, lon0, lat, lon):
    p0, p1 = np.radians(lat0), np.radians(lat)
    a = np.sin((p1 - p0) / 2) ** 2 + np.cos(p0) * np.cos(p1) * np.sin(np.radians(lon - lon0) / 2) ** 2
    return 2 * H.EARTH_RADIUS_KM * np.arcsin(np.sqrt(a))


fig, axes = plt.subplots(1, len(OUT), figsize=(6 * len(OUT), 4), squeeze=False)
for ax, (case, o) in zip(axes[0], OUT.items()):
    t = pd.Timestamp(CASES[case]["t_map"])
    i = int(np.where(pd.DatetimeIndex(o["track"]["time"]) == t)[0][0])
    row = o["track"].iloc[i]
    r = haversine_km(row["lat"], row["lon"], o["lat2d"], o["lon2d"]).ravel()
    bins = np.arange(0, 1200, 25)
    for exp, ls in (("v0.1", "--"), ("v0.2", "-")):
        hw = o["events"][exp]["H_W"].values[i].ravel()
        med = pd.Series(hw).groupby(np.clip(np.digitize(r, bins), 1, len(bins) - 1)).median()
        ax.plot(bins[med.index.to_numpy() - 1] + 12.5, med.values, ls, color=EXP_COLORS[exp],
                lw=2, label=f"{exp}")
    ax.axvline(float(np.nanmean([H.direct_radii_km(row)[q] for q in QUADRANTS])), color="0.5", lw=0.8)
    ax.axvline(float(row["pa21_r2_km"]), color="0.5", lw=0.8, ls=":")
    ax.set(xlabel="distancia al centro (km)", ylabel="mediana de H_W", title=f"{case}, {t:%m-%d %H} UTC",
           xlim=(0, 1150), ylim=(0, 1.02))
    ax.legend(frameon=False)
fig.suptitle("Viento: v0.1 cae a cero en R34 (línea gris); v0.2 decae hasta R_W06 (punteada)", fontsize=11)
fig.tight_layout()
fig.savefig(FIG_DIR / "perfil_radial_viento.png", dpi=200, bbox_inches="tight")

## 7. Área por clase a lo largo del evento

In [ ]:
def cell_area_km2(lat2d, lon2d):
    dlat = abs(float(np.diff(lat2d[:2, 0])[0])); dlon = abs(float(np.diff(lon2d[0, :2])[0]))
    return (np.radians(dlat) * H.EARTH_RADIUS_KM) * (np.radians(dlon) * H.EARTH_RADIUS_KM * np.cos(np.radians(lat2d)))


fig, axes = plt.subplots(2, len(OUT), figsize=(6 * len(OUT), 6), sharex="col", squeeze=False)
for j, (case, o) in enumerate(OUT.items()):
    area = cell_area_km2(o["lat2d"], o["lon2d"])
    for exp, ds in o["events"].items():
        x = ds["ITCHI"].values
        for k, (thr, lab) in enumerate(((0.4, "ITCHI > 0.4 (moderado o más)"), (0.8, "ITCHI > 0.8 (extremo)"))):
            axes[k, j].plot(ds["time"].values, ((x > thr) * area).sum(axis=(1, 2)) / 1e3, color=EXP_COLORS[exp],
                            lw=2, ls={"v0.1": "--", "R4": ":", "v0.2": "-"}[exp], label=exp)
            axes[k, j].set_ylabel(f"{lab}\n(10³ km²)")
    axes[0, j].set_title(case)
    axes[0, j].legend(frameon=False)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIG_DIR / "area_por_clase.png", dpi=200, bbox_inches="tight")

## 8. Verificación municipal contra declaratorias

Universo: municipios con al menos un píxel dentro de la región de atribución en algún snapshot. El valor de cada municipio es el máximo de `ITCHI_max` sobre sus píxeles. $D = 1$ si tuvo declaratoria asociada al ciclón (`CT_pico`), de cualquier tipo o solo de desastre. Una alerta es "clase SIAT ≥ k".

Con dos ciclones los intervalos son amplios: esto ilustra el efecto, la calibración es la del análisis nacional (`negativos_w06.py`).

In [ ]:
gdf = gpd.read_file(MUN_SHP)
gdf["CVEGEO"] = gdf["CVEGEO"].astype(str).str.zfill(5)
gdf = gdf.to_crs(4326) if gdf.crs is not None and gdf.crs.to_epsg() != 4326 else gdf
decl = pd.read_csv(DECL_FILE, dtype={"CVEGEO": str})


def municipal_table(case: str, o: dict) -> pd.DataFrame:
    b = o["bbox"]
    rp = gdf.geometry.representative_point()
    sub = gdf[rp.x.between(b["lon_min"], b["lon_max"]) & rp.y.between(b["lat_min"], b["lat_max"])]
    lat, lon = o["lat2d"][:, 0], o["lon2d"][0]
    pix = municipio_pixels(sub, lat, lon, sub["CVEGEO"])
    ev = o["events"]
    region = ((ev["v0.2"]["M_direct"] > 0) | (ev["v0.2"]["M_indirect"] > 0)).any("time").values.ravel()
    flat_max = {e: ds["ITCHI_max"].values.ravel() for e, ds in ev.items()}
    rows = []
    for code_, (flat, _) in pix.items():
        if not region[flat].any():
            continue
        rows.append({"CVEGEO": code_, **{e: float(np.nanmax(v[flat])) for e, v in flat_max.items()}})
    tab = pd.DataFrame(rows)
    d = decl[decl["CT_pico"] == CASES[case]["atcf"]]
    tab["D"] = tab["CVEGEO"].isin(d["CVEGEO"]).astype(int)
    tab["D_desastre"] = tab["CVEGEO"].isin(d.loc[d["Tipo Declaratoria"] == "Desastre", "CVEGEO"]).astype(int)
    for e in EXPERIMENTS:
        tab[f"clase_{e}"] = classify_itchi_to_siat_levels(tab[e].to_numpy())
    return tab


def scores(tab: pd.DataFrame, target: str = "D") -> pd.DataFrame:
    rows = []
    obs = tab[target].to_numpy(bool)
    keep = (tab["D"] == 0).to_numpy() | obs           # positivos de otro tipo fuera
    for e in EXPERIMENTS:
        lv = tab[f"clase_{e}"].to_numpy()
        for k in range(1, 6):
            p, y = (lv >= k)[keep], obs[keep]
            pod = (p & y).sum() / max(y.sum(), 1); pofd = (p & ~y).sum() / max((~y).sum(), 1)
            rows.append({"experimento": e, "clase >=": k, "POD": pod, "POFD": pofd, "PSS": pod - pofd})
    return pd.DataFrame(rows)


MUN, SCORES = {}, []
for case, o in OUT.items():
    MUN[case] = municipal_table(case, o)
    for target in ("D", "D_desastre"):
        s = scores(MUN[case], target).assign(caso=case, evento="cualquier declaratoria" if target == "D" else "desastre")
        SCORES.append(s)
    t = MUN[case]
    print(f"{case}: {len(t)} municipios en la zona | con declaratoria {int(t['D'].sum())} | desastre {int(t['D_desastre'].sum())}")
SCORES = pd.concat(SCORES, ignore_index=True)
SCORES.to_csv(FIG_DIR / "verificacion_municipal_casos.csv", index=False, float_format="%.3f")
SCORES.pivot_table(index=["caso", "evento", "clase >="], columns="experimento", values="PSS").round(3)

In [ ]:
fig, axes = plt.subplots(1, len(OUT), figsize=(6 * len(OUT), 3.8), sharey=True, squeeze=False)
for ax, case in zip(axes[0], OUT):
    s = SCORES[(SCORES["caso"] == case) & (SCORES["evento"] == "cualquier declaratoria")]
    w = 0.26
    for j, e in enumerate(EXPERIMENTS):
        v = s[s["experimento"] == e]
        ax.bar(v["clase >="] + (j - 1) * w, v["PSS"], w, color=EXP_COLORS[e], label=e,
               hatch={"v0.1": "//", "R4": "..", "v0.2": ""}[e], edgecolor="white")
    ax.axhline(0, color="0.4", lw=0.8)
    ax.set(xlabel="alerta si clase SIAT ≥", ylabel="PSS (POD − POFD)", title=case, xticks=range(1, 6))
    ax.legend(frameon=False)
fig.suptitle("Correspondencia municipal con declaratorias por experimento", fontsize=11)
fig.tight_layout()
fig.savefig(FIG_DIR / "pss_municipal_casos.png", dpi=200, bbox_inches="tight")

In [ ]:
for case, o in OUT.items():
    t = gdf.merge(MUN[case], on="CVEGEO")
    fig, axes = map_axes(o, 2)
    for ax, e in zip(axes[0], ("v0.1", "v0.2")):
        t.plot(ax=ax, column=f"clase_{e}", cmap=SIAT_CMAP, norm=SIAT_NORM, transform=PC, edgecolor="none")
        t[t["D"] == 1].boundary.plot(ax=ax, color="k", lw=0.5, transform=PC)
        ax.set_title(f"{e}: clase SIAT municipal (contorno = con declaratoria)", fontsize=10)
        b = o["bbox"]
        ax.set_extent([max(b["lon_min"], -118.5), min(b["lon_max"], -86.0), max(b["lat_min"], 14.0),
                       min(b["lat_max"], 33.5)], crs=PC)
    fig.savefig(FIG_DIR / f"{case.lower()}_municipios_v01_v02.png", dpi=200, bbox_inches="tight")

## 9. Resumen para la presentación

In [ ]:
rows = []
for case, o in OUT.items():
    area = cell_area_km2(o["lat2d"], o["lon2d"])
    for e, ds in o["events"].items():
        x = np.nan_to_num(ds["ITCHI_max"].values)
        s = SCORES[(SCORES["caso"] == case) & (SCORES["experimento"] == e) & (SCORES["evento"] == "cualquier declaratoria")]
        rows.append({"caso": case, "experimento": e,
                     "área ITCHI_max > 0.4 (10³ km²)": (area * (x > 0.4)).sum() / 1e3,
                     "área ITCHI_max > 0.8 (10³ km²)": (area * (x > 0.8)).sum() / 1e3,
                     "PSS máx. municipal": s["PSS"].max(),
                     "clase del PSS máx.": int(s.loc[s["PSS"].idxmax(), "clase >="])})
RESUMEN = pd.DataFrame(rows)
RESUMEN.to_csv(FIG_DIR / "resumen_casos.csv", index=False, float_format="%.3f")
print("figuras y tablas en", FIG_DIR)
RESUMEN.round(3)